In [0]:
WITH periods AS (
    SELECT 'YTD 2026' AS Period, DATE'2026-01-01' AS start_date, DATE'2026-08-31' AS end_date
    UNION ALL
    SELECT 'YTD 2025', DATE'2025-01-01', DATE'2025-08-31'
),

ticket_events_classified AS (
    SELECT
        t.CDE_ID AS cde_id,
        d.datefull,
        t.LocationId,
        t.CineClubFLAG AS cineclub_flag,
        t.SCENEFLAG AS scene_flag,
        t.TicketFLAG AS ticket_flag,
        t.ConcessionFLAG AS concession_flag,
        t.Quantity AS quantity,
        t.Net_Revenue AS net_revenue,
        CASE
            WHEN s.Auditorium_Experience_ID = 2      THEN 'AVX'
            WHEN s.Auditorium_Experience_ID = 3      THEN 'IMAX'
            WHEN s.Auditorium_Experience_ID = 4      THEN 'VIP'
            WHEN s.Auditorium_Experience_ID IN (5,6) THEN 'ALT'
            WHEN s.Auditorium_Experience_ID = 7      THEN 'FourDX'
            WHEN s.Auditorium_Experience_ID = 8      THEN 'ScreenX'
            WHEN s.Performance_Experience_ID = 2     THEN 'ThreeD'
            ELSE 'Regular'
        END AS final_format
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
    LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_session s ON t.sessionskey = s.sessionskey
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
),

visit_grain_cc AS (
    SELECT
        cde_id, datefull, LocationId,
        SUM(quantity) AS Net_Quantity,
        SUM(CASE WHEN ticket_flag = 'Yes' THEN quantity ELSE 0 END) AS Net_Ticket_Quantity,
        SUM(CASE WHEN ticket_flag = 'Yes' THEN net_revenue ELSE 0 END) AS Net_Ticket_Spend,
        SUM(CASE WHEN ticket_flag = 'Yes' AND final_format != 'Regular' THEN quantity ELSE 0 END) AS Premium_Ticket_Quantity,
        SUM(CASE WHEN concession_flag = 'Yes' THEN quantity ELSE 0 END) AS Net_Concession_Quantity,
        SUM(CASE WHEN concession_flag = 'Yes' THEN net_revenue ELSE 0 END) AS Net_Concession_Spend,
        MAX(CASE WHEN concession_flag = 'Yes' AND quantity > 0 THEN 1 ELSE 0 END) AS has_concession_visit
    FROM ticket_events_classified
    WHERE cde_id IS NOT NULL AND cde_id > 0 AND cineclub_flag = 'Yes'
    GROUP BY cde_id, datefull, LocationId
    HAVING SUM(quantity) > 0
),

visit_grain_scene_non_cc AS (
    SELECT
        cde_id, datefull, LocationId,
        SUM(quantity) AS Net_Quantity,
        SUM(CASE WHEN ticket_flag = 'Yes' THEN quantity ELSE 0 END) AS Net_Ticket_Quantity,
        SUM(CASE WHEN ticket_flag = 'Yes' THEN net_revenue ELSE 0 END) AS Net_Ticket_Spend,
        SUM(CASE WHEN ticket_flag = 'Yes' AND final_format != 'Regular' THEN quantity ELSE 0 END) AS Premium_Ticket_Quantity,
        SUM(CASE WHEN concession_flag = 'Yes' THEN quantity ELSE 0 END) AS Net_Concession_Quantity,
        SUM(CASE WHEN concession_flag = 'Yes' THEN net_revenue ELSE 0 END) AS Net_Concession_Spend,
        MAX(CASE WHEN concession_flag = 'Yes' AND quantity > 0 THEN 1 ELSE 0 END) AS has_concession_visit
    FROM ticket_events_classified
    WHERE cde_id IS NOT NULL AND cde_id > 0 AND scene_flag = 'Yes' AND cineclub_flag != 'Yes'
    GROUP BY cde_id, datefull, LocationId
    HAVING SUM(quantity) > 0
),

visit_grain_base AS (
    SELECT 'CineClub' AS Cohort, * FROM visit_grain_cc
    UNION ALL
    SELECT 'SCENE+ (Non-CineClub)' AS Cohort, * FROM visit_grain_scene_non_cc
),

period_member_metrics AS (
    SELECT
        p.Period,
        vgb.Cohort,
        vgb.cde_id,
        COUNT(DISTINCT vgb.datefull) AS Visits_In_Selected_Period,
        SUM(vgb.Net_Ticket_Quantity) AS Tickets_In_Selected_Period,
        SUM(vgb.Premium_Ticket_Quantity) AS Premium_Tickets_In_Selected_Period,
        SUM(vgb.Net_Ticket_Spend) AS Ticket_Spend_In_Selected_Period,
        SUM(vgb.Net_Concession_Quantity) AS Concessions_In_Selected_Period,
        SUM(vgb.Net_Concession_Spend) AS Concession_Spend_In_Selected_Period,
        SUM(vgb.has_concession_visit) AS Concession_Visits_In_Selected_Period
    FROM periods p
    JOIN visit_grain_base vgb ON vgb.datefull BETWEEN p.start_date AND p.end_date
    GROUP BY p.Period, vgb.Cohort, vgb.cde_id
),

active_members AS (
    SELECT * FROM period_member_metrics WHERE Visits_In_Selected_Period > 0
),

cohort_period_summary AS (
    SELECT
        Period,
        Cohort,
        COUNT(DISTINCT cde_id) AS Members_with_1_plus_visits,
        SUM(Visits_In_Selected_Period) AS Total_Visits,
        SUM(Tickets_In_Selected_Period) AS Total_Tickets,
        SUM(Premium_Tickets_In_Selected_Period) AS Premium_Tickets_Purchased,
        SUM(Ticket_Spend_In_Selected_Period) AS Total_Ticket_Spend,
        SUM(Concession_Spend_In_Selected_Period) AS Total_Concession_Spend,
        SUM(Concession_Visits_In_Selected_Period) AS Concession_Visits
    FROM active_members
    GROUP BY Period, Cohort
),

-- Numerator: "999 tickets" (CineClub_Discount_Type_ID 5 = companion, 6 = free redeemed)
redeemed_999_by_period AS (
    SELECT p.Period, SUM(agg.qty) AS Total_999_Tickets
    FROM periods p
    JOIN (
        SELECT t.CDE_ID AS cde_id, d.datefull, t.LocationId, SUM(t.Quantity) AS qty
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
        WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
          AND t.CineClub_Discount_Type_ID IN (5,6,7,8)
        GROUP BY t.CDE_ID, d.datefull, t.LocationId
    ) agg ON agg.datefull BETWEEN p.start_date AND p.end_date
    GROUP BY p.Period
),

-- Denominator: total paid ticket attendance, all patrons, no cohort filter
paid_tickets_by_period AS (
    SELECT p.Period, SUM(agg.net_qty) AS Total_Paid_Tickets
    FROM periods p
    JOIN (
        SELECT t.CDE_ID AS cde_id, d.datefull, t.LocationId, t.sessionskey, t.TicketTypeSkey,
               SUM(t.Quantity) AS net_qty
        FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc ON t.LocationId = loc.LocationID
        JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d ON t.VisitDateId = d.dateid
        LEFT JOIN cpx_dataanalytics_gold.customer360.cpx_d_ticket_type tk ON t.TicketTypeSkey = tk.tickettypeskey
        WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
          AND t.TicketFLAG = 'Yes'
          AND tk.paid_attendance_inflag = 'Yes'
        GROUP BY t.CDE_ID, d.datefull, t.LocationId, t.sessionskey, t.TicketTypeSkey
        HAVING SUM(t.Quantity) > 0
    ) agg ON agg.datefull BETWEEN p.start_date AND p.end_date
    GROUP BY p.Period
)

SELECT
    s.Period,
    s.Cohort,
    s.Members_with_1_plus_visits                                              AS `Members with 1+ visit`,
    s.Total_Visits                                                            AS `Total Visits`,
    ROUND(s.Total_Visits * 1.0 / s.Members_with_1_plus_visits, 2)             AS `Visits/Member`,
    s.Total_Tickets                                                           AS `Total Tickets Purchased`,
    ROUND(s.Total_Tickets * 1.0 / s.Members_with_1_plus_visits, 2)            AS `Tickets/Member`,
    ROUND(s.Total_Tickets * 1.0 / s.Total_Visits, 2)                          AS `Tickets/Visit`,
    s.Premium_Tickets_Purchased                                               AS `Premium Tickets Purchased`,
    ROUND(s.Premium_Tickets_Purchased * 100.0 / NULLIF(s.Total_Tickets,0), 2) AS `% Upgraded`,
    ROUND(s.Total_Ticket_Spend / s.Members_with_1_plus_visits, 2)             AS `Ticket Spend/Member`,
    ROUND(s.Total_Concession_Spend / s.Members_with_1_plus_visits, 2)         AS `Concession Spend/Member`,
    s.Concession_Visits                                                       AS `Concession Visits`,
    ROUND((s.Total_Ticket_Spend + s.Total_Concession_Spend) / s.Members_with_1_plus_visits, 2) AS `Total Spend/Member`,
    CASE WHEN s.Cohort = 'CineClub'
         THEN ROUND(r.Total_999_Tickets * 100.0 / NULLIF(pt.Total_Paid_Tickets, 0), 2)
         ELSE NULL END                                                        AS `% of EXH Attendance`
FROM cohort_period_summary s
LEFT JOIN redeemed_999_by_period r ON r.Period = s.Period
LEFT JOIN paid_tickets_by_period pt ON pt.Period = s.Period
ORDER BY s.Period, s.Cohort;